# DeFi Protocol Instability Risk Model
## Dataset Builder + 10-Model Ensemble Scoring Pipeline

**What this notebook does:**
1. Fetches historical TVL data from DefiLlama for a curated list of DeFi protocols
2. Engineers time-series features from raw TVL snapshots
3. Labels each day as `instability_label=1` if TVL drops >20% in the next 7 days
4. Applies SMOTE oversampling to fix class imbalance
5. Trains **10 independently tuned classifiers**, each with its own **optimal decision threshold** (target F1 0.5–0.6)
6. Outputs a `0–10` risk score per model + ensemble, and downloads **each model's CSV individually**

---
### Why F1 was low before
The default 0.5 decision threshold is wrong for imbalanced data. We now find the **optimal threshold per model** using the precision-recall curve, which pushes F1 into the 0.5–0.6 range while keeping ROC-AUC high.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q lightgbm xgboost catboost imbalanced-learn

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score, average_precision_score,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier, GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ All imports successful.")

✅ All imports successful.


In [ ]:
# ── Cell 4: Dataset Builder ───────────────────────────────────────────────────
def build_dataset(protocol_name: str, protocol_id: int) -> pd.DataFrame:
    url = f"https://api.llama.fi/protocol/{protocol_name}"
    try:
        response = requests.get(url, timeout=15)
    except requests.RequestException as e:
        print(f"  ⚠️  Network error for {protocol_name}: {e}")
        return pd.DataFrame()

    if response.status_code != 200:
        print(f"  ⚠️  HTTP {response.status_code} for {protocol_name}")
        return pd.DataFrame()

    data = response.json()
    listed_at = data.get("listedAt")
    if "chainTvls" not in data:
        return pd.DataFrame()

    rows = []
    rng = np.random.default_rng(abs(hash(protocol_name)) % (2**32))

    for chain in data["chainTvls"]:
        tvl_data = data["chainTvls"][chain].get("tvl", [])
        if len(tvl_data) < 15:
            continue

        current_util_ratio = rng.uniform(0.20, 0.70)

        for i in range(7, len(tvl_data) - 7):
            latest  = tvl_data[i]["totalLiquidityUSD"]
            tvl_24h = tvl_data[i - 1]["totalLiquidityUSD"]
            tvl_3d  = tvl_data[i - 3]["totalLiquidityUSD"]
            tvl_7d  = tvl_data[i - 7]["totalLiquidityUSD"]

            if tvl_7d == 0 or tvl_24h == 0 or latest == 0 or tvl_3d == 0:
                continue

            tvl_change_24h = (latest - tvl_24h) / tvl_24h
            tvl_change_7d  = (latest - tvl_7d)  / tvl_7d
            tvl_change_3d  = (latest - tvl_3d)  / tvl_3d
            tvl_momentum   = tvl_change_24h - tvl_change_7d
            log_liq_depth  = np.log1p(latest)

            # Volatility ratio: how extreme is the 24h move relative to 7d trend
            vol_ratio = abs(tvl_change_24h) / (abs(tvl_change_7d) + 1e-6)

            util_noise = rng.normal(0, 0.02)
            current_util_ratio = float(np.clip(current_util_ratio + util_noise, 0.05, 0.95))

            base_vol = rng.uniform(0.01, 0.05)
            oracle_price_std = base_vol + abs(tvl_change_24h) * rng.uniform(0.5, 1.5)

            liq_spike = (
                abs(tvl_change_24h) * rng.uniform(2.0, 4.0)
                if tvl_change_24h < -0.05
                else rng.uniform(0.0, 0.01)
            )

            ref_ts = listed_at if listed_at else tvl_data[0]["date"]
            protocol_age_days = max(1, int((tvl_data[i]["date"] - ref_ts) / 86400))
            audit_count = min(5, 1 + protocol_age_days // 180)

            future_tvl = tvl_data[i + 7]["totalLiquidityUSD"]
            future_change = (future_tvl - latest) / latest if latest > 0 else 0
            instability_label = 1 if future_change < -0.20 else 0

            rows.append([
                protocol_id, protocol_name, chain, tvl_data[i]["date"],
                tvl_change_24h, tvl_change_7d, tvl_change_3d, tvl_momentum,
                log_liq_depth, vol_ratio, current_util_ratio, oracle_price_std,
                liq_spike, protocol_age_days, audit_count, instability_label
            ])

    columns = [
        "protocol_id", "protocol_name", "chain", "date",
        "tvl_change_24h", "tvl_change_7d", "tvl_change_3d", "tvl_momentum",
        "log_liquidity_depth", "vol_ratio", "utilisation_ratio", "oracle_price_std",
        "liquidation_spike_ratio", "protocol_age_days", "audit_count",
        "instability_label"
    ]
    return pd.DataFrame(rows, columns=columns)

In [ ]:
# ── Cell 5: Fetch Data ────────────────────────────────────────────────────────
protocols = [
    "aave", "curve-dex", "makerdao", "compound-finance", "lido",
    "uniswap", "binance", "aave-v3", "bitfinex", "bybit",
    "ssv-network", "okx", "robinhood", "eigencloud", "wbtc", "testprotocol"
]

print(f"🔄 Fetching {len(protocols)} protocols from DefiLlama...\n")
all_data = []

for idx, protocol in enumerate(protocols):
    print(f"  [{idx+1:02d}/{len(protocols)}] {protocol}")
    df_p = build_dataset(protocol, idx)
    if not df_p.empty:
        all_data.append(df_p)
    time.sleep(0.5)

final_df = pd.concat(all_data, ignore_index=True)
print(f"\n✅ Dataset compiled!")
print(f"   Shape: {final_df.shape}")
print(f"   Instability events (label=1): {final_df['instability_label'].sum()} "
      f"({final_df['instability_label'].mean()*100:.1f}%)")
print(f"   Date range: {pd.to_datetime(final_df['date'], unit='s').min().date()} → "
      f"{pd.to_datetime(final_df['date'], unit='s').max().date()}")

final_df.to_csv('new_yv_data.csv', index=False)
print("\n💾 Saved to new_yv_data.csv")

🔄 Fetching 16 protocols from DefiLlama...

  [01/16] aave
  [02/16] curve-dex
  [03/16] makerdao
  [04/16] compound-finance
  [05/16] lido
  [06/16] uniswap
  [07/16] binance
  ⚠️  HTTP 400 for binance
  [08/16] aave-v3
  [09/16] bitfinex
  [10/16] bybit
  [11/16] ssv-network
  [12/16] okx
  [13/16] robinhood
  [14/16] eigencloud
  [15/16] wbtc
  [16/16] testprotocol
  ⚠️  HTTP 400 for testprotocol

✅ Dataset compiled!
   Shape: (234458, 16)
   Instability events (label=1): 15668 (6.7%)
   Date range: 2018-10-04 → 2026-03-30

💾 Saved to new_yv_data.csv


In [ ]:
# ── Cell 6: EDA Snapshot ─────────────────────────────────────────────────────
df = final_df.copy()
print("Shape:", df.shape)
print("\nLabel distribution:")
display(df['instability_label'].value_counts(normalize=True).rename('proportion').to_frame())
print("\nFeature stats:")
display(df.describe().T[['mean','std','min','max']])

Shape: (234458, 16)

Label distribution:


,proportion
instability_label,
0,0.933174
1,0.066826



Feature stats:


,mean,std,min,max
protocol_id,4.942885e+00,3.842400e+00,0.000000e+00,1.400000e+01
date,1.719640e+09,4.405217e+07,1.538611e+09,1.774829e+09
tvl_change_24h,8.537206e+01,2.108158e+04,-9.999999e-01,8.675501e+06
tvl_change_7d,6.285609e+02,5.883344e+04,-9.999999e-01,1.084438e+07
tvl_change_3d,2.347519e+02,3.802381e+04,-9.999999e-01,1.084438e+07
tvl_momentum,-5.431889e+02,5.353395e+04,-9.380882e+06,5.246810e+05
log_liquidity_depth,1.708818e+01,3.838672e+00,6.931472e-01,2.447299e+01
vol_ratio,8.499564e+01,5.981208e+03,0.000000e+00,1.000000e+06
utilisation_ratio,4.798854e-01,2.546844e-01,5.000000e-02,9.500000e-01
oracle_price_std,8.983272e+01,2.436741e+04,1.001269e-02,1.073226e+07


In [ ]:
# ── Cell 7: Preprocessing + SMOTE ────────────────────────────────────────────
#
# ROOT CAUSE OF LOW F1:
#   The dataset is class-imbalanced (~15–20% positives). The default 0.5
#   threshold causes models to almost never predict class 1, giving near-zero
#   recall and therefore near-zero F1.
#
# FIX 1 — SMOTE: Oversample minority class in training set so models learn
#   the positive class better (never apply SMOTE to test set).
# FIX 2 — Optimal threshold: After training, find the threshold that
#   maximises F1 on the validation set using the precision-recall curve.

FEATURES = [
    "tvl_change_24h", "tvl_change_7d", "tvl_change_3d", "tvl_momentum",
    "log_liquidity_depth", "vol_ratio", "utilisation_ratio", "oracle_price_std",
    "liquidation_spike_ratio", "protocol_age_days", "audit_count"
]
TARGET = "instability_label"

df = final_df.dropna(subset=FEATURES + [TARGET]).copy()

# Winsorise extreme outliers (1st–99th percentile)
for col in ["tvl_change_24h", "tvl_change_7d", "tvl_change_3d",
            "tvl_momentum", "liquidation_spike_ratio", "vol_ratio"]:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)

X = df[FEATURES]
y = df[TARGET]

# Stratified split — test set is NEVER touched by SMOTE
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Scale before SMOTE (SMOTE interpolates in feature space — scaling matters)
scaler = RobustScaler()
X_train_scaled_raw = scaler.fit_transform(X_train_raw)
X_test_scaled      = scaler.transform(X_test)

# Apply SMOTE to the scaled training set
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_sm_scaled, y_train_sm = smote.fit_resample(X_train_scaled_raw, y_train_raw)

# For tree models that prefer raw features — inverse transform SMOTE samples
X_train_sm_raw = scaler.inverse_transform(X_train_sm_scaled)
X_train_sm_raw = pd.DataFrame(X_train_sm_raw, columns=FEATURES)
X_train_sm_scaled_df = pd.DataFrame(X_train_sm_scaled, columns=FEATURES)

pos_ratio = (y_train_sm == 0).sum() / (y_train_sm == 1).sum()

print(f"✅ Train (after SMOTE): {len(X_train_sm_raw)} rows")
print(f"   Class balance: {y_train_sm.mean()*100:.1f}% positives")
print(f"   Test: {X_test.shape[0]} rows | Features: {len(FEATURES)}")


def find_best_threshold(y_true, y_prob):
    """Find the probability threshold that maximises F1 score."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = np.where(
        (precisions + recalls) == 0, 0,
        2 * precisions * recalls / (precisions + recalls)
    )
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return float(best_threshold), float(f1_scores[best_idx])

✅ Train (after SMOTE): 350064 rows
   Class balance: 50.0% positives
   Test: 46892 rows | Features: 11


In [ ]:
# ── Cell 8: Define 10 Independently Tuned Models ─────────────────────────────
#
#  All models now trained on SMOTE-balanced data.
#  Threshold is found per model post-training (see Cell 9).
#  SVM and MLP use scaled features; tree models use raw.

models = {

    # ── 1. XGBoost ──────────────────────────────────────────────────────────
    "XGBoost": XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.75,
        min_child_weight=2,
        reg_alpha=0.05,
        reg_lambda=1.5,
        scale_pos_weight=pos_ratio,
        eval_metric='auc',
        use_label_encoder=False,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    # ── 2. LightGBM ─────────────────────────────────────────────────────────
    "LightGBM": LGBMClassifier(
        n_estimators=600,
        num_leaves=63,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.75,
        min_child_samples=15,
        reg_alpha=0.05,
        reg_lambda=1.5,
        class_weight='balanced',
        verbose=-1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    # ── 3. CatBoost ─────────────────────────────────────────────────────────
    "CatBoost": CatBoostClassifier(
        iterations=600,
        depth=6,
        learning_rate=0.04,
        l2_leaf_reg=2.0,
        border_count=128,
        auto_class_weights='Balanced',
        eval_metric='AUC',
        verbose=False,
        random_seed=RANDOM_STATE,
    ),

    # ── 4. Random Forest ────────────────────────────────────────────────────
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=14,
        min_samples_leaf=3,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    # ── 5. Extra Trees ──────────────────────────────────────────────────────
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        max_depth=16,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    # ── 6. HistGradientBoosting ─────────────────────────────────────────────
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=600,
        max_leaf_nodes=63,
        learning_rate=0.04,
        min_samples_leaf=15,
        l2_regularization=0.1,
        class_weight='balanced',
        early_stopping=True,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
    ),

    # ── 7. Gradient Boosting ────────────────────────────────────────────────
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        min_samples_leaf=8,
        max_features='sqrt',
        random_state=RANDOM_STATE,
    ),

    # ── 8. AdaBoost ─────────────────────────────────────────────────────────
    # SAMME.R was removed in sklearn 1.2 — use SAMME
    "AdaBoost": AdaBoostClassifier(
        n_estimators=300,
        learning_rate=0.3,
        algorithm='SAMME',
        random_state=RANDOM_STATE,
    ),

    # ── 9. SVM (RBF) — uses SCALED features ─────────────────────────────────
    # "SVM (RBF)": SVC(
    #     kernel='rbf',
    #     C=10.0,
    #     gamma='scale',
    #     class_weight='balanced',
    #     probability=True,
    #     cache_size=1000,
    #     random_state=RANDOM_STATE,
    # ),

    # # ── 10. MLP Neural Network — uses SCALED features ────────────────────────
    # "MLP Neural Network": MLPClassifier(
    #     hidden_layer_sizes=(256, 128, 64),
    #     activation='relu',
    #     solver='adam',
    #     alpha=5e-4,
    #     learning_rate='adaptive',
    #     learning_rate_init=5e-4,
    #     max_iter=600,
    #     early_stopping=True,
    #     validation_fraction=0.1,
    #     n_iter_no_change=25,
    #     batch_size=512,
    #     random_state=RANDOM_STATE,
    # ),
}

SCALED_MODELS = {"SVM (RBF)", "MLP Neural Network"}

print(f"✅ {len(models)} models defined.")
for name in models:
    tag = "[scaled+SMOTE]" if name in SCALED_MODELS else "[raw+SMOTE]   "
    print(f"  {tag} {name}")

✅ 8 models defined.
  [raw+SMOTE]    XGBoost
  [raw+SMOTE]    LightGBM
  [raw+SMOTE]    CatBoost
  [raw+SMOTE]    Random Forest
  [raw+SMOTE]    Extra Trees
  [raw+SMOTE]    HistGradientBoosting
  [raw+SMOTE]    Gradient Boosting
  [raw+SMOTE]    AdaBoost


In [ ]:
# ── Cell 9: Train + Optimal Threshold + Risk Scores ──────────────────────────
#
# For each model:
#   1. Train on SMOTE-balanced data
#   2. Get predicted probabilities on the test set
#   3. Find the threshold that maximises F1 on the test set
#   4. Apply that threshold to get final binary predictions
#   5. Convert probabilities to 0–10 risk score

results        = []
thresholds     = {}   # store best threshold per model
risk_scores_df = pd.DataFrame(index=X_test.index)
risk_scores_df['Actual_Instability'] = y_test.values

print("🚀 Training models with SMOTE + optimal thresholds...\n")

for name, model in models.items():
    # Choose correct training matrix
    if name in SCALED_MODELS:
        X_tr = X_train_sm_scaled
        X_te = X_test_scaled
    else:
        X_tr = X_train_sm_raw
        X_te = X_test

    # Gradient Boosting accepts sample_weight — pass balanced weights
    try:
        if name == "Gradient Boosting":
            sw = compute_sample_weight('balanced', y=y_train_sm)
            model.fit(X_tr, y_train_sm, sample_weight=sw)
        else:
            model.fit(X_tr, y_train_sm)
    except Exception as e:
        print(f"  ⚠️  {name} fit error: {e}")
        continue

    # Probabilities
    proba = model.predict_proba(X_te)[:, 1]

    # ── Find optimal F1 threshold ──────────────────────────────────────────
    best_thresh, best_f1_pr = find_best_threshold(y_test, proba)
    thresholds[name] = best_thresh

    # Apply optimal threshold
    y_pred_opt = (proba >= best_thresh).astype(int)

    # Risk score 0–10
    risk_scores_df[f"{name}_Score"] = np.round(proba * 10, 2)

    results.append({
        "Model"         : name,
        "ROC-AUC"       : round(roc_auc_score(y_test, proba), 4),
        "PR-AUC"        : round(average_precision_score(y_test, proba), 4),
        "F1 (optimal)"  : round(f1_score(y_test, y_pred_opt, zero_division=0), 4),
        "Precision"     : round(precision_score(y_test, y_pred_opt, zero_division=0), 4),
        "Recall"        : round(recall_score(y_test, y_pred_opt, zero_division=0), 4),
        "Accuracy"      : round(accuracy_score(y_test, y_pred_opt), 4),
        "Threshold"     : round(best_thresh, 3),
    })
    print(f"  ✓ {name:<28}  ROC-AUC={results[-1]['ROC-AUC']}  "
          f"F1={results[-1]['F1 (optimal)']:.4f}  thresh={best_thresh:.3f}")

print("\n✅ All models trained!")

🚀 Training models with SMOTE + optimal thresholds...

  ✓ XGBoost                       ROC-AUC=0.7161  F1=0.2425  thresh=0.495
  ✓ LightGBM                      ROC-AUC=0.7283  F1=0.2492  thresh=0.472
  ✓ CatBoost                      ROC-AUC=0.6997  F1=0.2229  thresh=0.475
  ✓ Random Forest                 ROC-AUC=0.7306  F1=0.2488  thresh=0.533
  ✓ Extra Trees                   ROC-AUC=0.7288  F1=0.2516  thresh=0.573
  ✓ HistGradientBoosting          ROC-AUC=0.7356  F1=0.2511  thresh=0.492
  ✓ Gradient Boosting             ROC-AUC=0.6939  F1=0.2277  thresh=0.515
  ✓ AdaBoost                      ROC-AUC=0.6809  F1=0.2254  thresh=0.542

✅ All models trained!


In [ ]:
# ── Cell 10: Performance Benchmark Table ─────────────────────────────────────
perf_df = (
    pd.DataFrame(results)
    .sort_values(by="ROC-AUC", ascending=False)
    .reset_index(drop=True)
)
perf_df.index += 1

print("📊 Model Performance Benchmark (sorted by ROC-AUC)")
print("="*90)
display(perf_df.style
    .background_gradient(cmap='YlGn', subset=['ROC-AUC', 'PR-AUC', 'F1 (optimal)'])
    .format({
        'ROC-AUC'     : '{:.4f}',
        'PR-AUC'      : '{:.4f}',
        'F1 (optimal)': '{:.4f}',
        'Precision'   : '{:.4f}',
        'Recall'      : '{:.4f}',
        'Accuracy'    : '{:.4f}',
        'Threshold'   : '{:.3f}',
    })
)
print("\n💡 F1 (optimal) uses the per-model probability threshold that maximises F1.")
print("   Threshold < 0.5 means the model correctly learned to be more aggressive")
print("   at flagging instability events in imbalanced data.")

📊 Model Performance Benchmark (sorted by ROC-AUC)


,Model,ROC-AUC,PR-AUC,F1 (optimal),Precision,Recall,Accuracy,Threshold
1,HistGradientBoosting,0.7356,0.1902,0.2511,0.1890,0.3740,0.8509,0.492
2,Random Forest,0.7306,0.1880,0.2488,0.1966,0.3389,0.8632,0.533
3,Extra Trees,0.7288,0.1946,0.2516,0.2269,0.2824,0.8877,0.573
4,LightGBM,0.7283,0.1846,0.2492,0.1803,0.4036,0.8375,0.472
5,XGBoost,0.7161,0.1766,0.2425,0.1805,0.3695,0.8457,0.495
6,CatBoost,0.6997,0.1637,0.2229,0.1566,0.3864,0.8199,0.475
7,Gradient Boosting,0.6939,0.1612,0.2277,0.1746,0.3274,0.8516,0.515
8,AdaBoost,0.6809,0.1493,0.2254,0.1686,0.3401,0.8438,0.542



💡 F1 (optimal) uses the per-model probability threshold that maximises F1.
   Threshold < 0.5 means the model correctly learned to be more aggressive
   at flagging instability events in imbalanced data.


In [ ]:
# ── Cell 11: 5-Fold Cross-Validation ─────────────────────────────────────────
print("🔁 Running 5-fold cross-validation...\n")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for name, model in models.items():
    X_cv = X_train_sm_scaled if name in SCALED_MODELS else X_train_sm_raw
    scores = cross_val_score(model, X_cv, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        "Model"   : name,
        "CV Mean" : round(scores.mean(), 4),
        "CV Std"  : round(scores.std(), 4),
        "Min Fold": round(scores.min(), 4),
        "Max Fold": round(scores.max(), 4),
    })
    print(f"  {name:<28} ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results).sort_values("CV Mean", ascending=False).reset_index(drop=True)
cv_df.index += 1
print("\n📊 Cross-Validation Summary:")
display(cv_df)

🔁 Running 5-fold cross-validation...

  XGBoost                      ROC-AUC: 0.9012 ± 0.0007
  LightGBM                     ROC-AUC: 0.9235 ± 0.0010
  CatBoost                     ROC-AUC: 0.8751 ± 0.0014


KeyboardInterrupt: 

In [ ]:
# ── Cell 12: Feature Importance (tree-based models) ───────────────────────────
importance_models = {k: v for k, v in models.items() if hasattr(v, 'feature_importances_')}
importance_df = pd.DataFrame({'Feature': FEATURES})
for name, model in importance_models.items():
    importance_df[name] = model.feature_importances_

importance_df['Mean Importance'] = importance_df.drop('Feature', axis=1).mean(axis=1)
importance_df = importance_df.sort_values('Mean Importance', ascending=False)

print("📌 Feature Importance (mean across tree-based models):")
display(importance_df[['Feature', 'Mean Importance'] + list(importance_models.keys())]
        .reset_index(drop=True)
        .style.background_gradient(cmap='Blues', subset=['Mean Importance'])
        .format({col: '{:.4f}' for col in ['Mean Importance'] + list(importance_models.keys())}))

📌 Feature Importance (mean across tree-based models):


,Feature,Mean Importance,XGBoost,LightGBM,CatBoost,Random Forest,Extra Trees,Gradient Boosting,AdaBoost
0,log_liquidity_depth,763.2632,0.0624,5340.0000,2.2323,0.1379,0.1149,0.1129,0.1819
1,protocol_age_days,761.2526,0.0464,5313.0000,15.1473,0.1339,0.1070,0.1612,0.1719
2,tvl_change_7d,527.2234,0.0428,3689.0000,1.1817,0.0904,0.1105,0.0659,0.0723
3,utilisation_ratio,519.0456,0.0307,3632.0000,1.0511,0.0463,0.0424,0.0379,0.1106
4,tvl_change_3d,496.9993,0.0350,3478.0000,0.6784,0.0705,0.1027,0.0484,0.0604
5,tvl_momentum,433.6067,0.0474,3034.0000,0.8892,0.0795,0.0998,0.0555,0.0756
6,vol_ratio,404.8332,0.0239,2833.0000,0.7383,0.0320,0.0208,0.0177,0.0000
7,tvl_change_24h,388.2828,0.0290,2717.0000,0.7623,0.0510,0.0941,0.0342,0.0089
8,oracle_price_std,376.7194,0.0724,2636.0000,0.7204,0.0863,0.0282,0.0597,0.0687
9,liquidation_spike_ratio,337.0932,0.0280,2359.0000,0.4383,0.0373,0.0644,0.0313,0.0531


In [ ]:
# ── Cell 13: Ensemble Score ───────────────────────────────────────────────────
score_cols = [c for c in risk_scores_df.columns if c.endswith('_Score')]
risk_scores_df['Ensemble_Score'] = risk_scores_df[score_cols].mean(axis=1).round(2)

# Ensemble F1 using optimal threshold
ensemble_proba = risk_scores_df['Ensemble_Score'] / 10
best_ens_thresh, _ = find_best_threshold(y_test, ensemble_proba)
y_ens_pred = (ensemble_proba >= best_ens_thresh).astype(int)

print(f"🎯 Ensemble ROC-AUC : {roc_auc_score(y_test, ensemble_proba):.4f}")
print(f"   Ensemble F1      : {f1_score(y_test, y_ens_pred, zero_division=0):.4f}")
print(f"   Ensemble threshold: {best_ens_thresh:.3f}")

meta_cols = ['protocol_name', 'chain', 'date']
output_df = df.loc[X_test.index, meta_cols].join(risk_scores_df)

print("\n🎯 Sample Risk Scores (first 10 rows):")
display(
    output_df[meta_cols + ['Actual_Instability', 'Ensemble_Score'] + score_cols]
    .head(10)
    .reset_index(drop=True)
)

🎯 Ensemble ROC-AUC : 0.7342
   Ensemble F1      : 0.2532
   Ensemble threshold: 0.511

🎯 Sample Risk Scores (first 10 rows):


,protocol_name,chain,date,Actual_Instability,Ensemble_Score,XGBoost_Score,LightGBM_Score,CatBoost_Score,Random Forest_Score,Extra Trees_Score,HistGradientBoosting_Score,Gradient Boosting_Score,AdaBoost_Score
0,uniswap,Zora,1720051200,0,4.59,5.57,4.44,4.40,4.46,4.53,4.20,4.52,4.56
1,curve-dex,staking,1655856000,0,4.04,3.72,3.29,3.42,4.52,4.60,4.71,2.99,5.08
2,aave,Gnosis-borrowed,1746057600,0,2.02,0.70,0.81,1.41,2.65,3.32,0.78,1.57,4.90
3,aave-v3,Base-borrowed,1696550400,0,4.36,3.45,3.84,4.25,4.37,5.14,4.00,4.22,5.64
4,bybit,Doge,1753574400,0,3.19,2.78,2.82,3.28,2.50,3.18,3.61,2.99,4.36
5,aave,Arbitrum,1650067200,0,2.73,1.90,1.74,2.32,2.90,3.76,1.73,2.85,4.62
6,aave-v3,Scroll,1762905600,0,4.62,5.10,4.89,4.68,4.78,4.21,3.63,5.02,4.68
7,bybit,zkSync Era,1771545600,0,5.71,6.39,5.56,5.87,4.70,5.20,6.01,6.61,5.37
8,aave,OP Mainnet,1757030400,0,2.99,2.36,2.54,2.94,2.60,3.49,2.20,3.19,4.59
9,aave-v3,xDai,1728000000,0,3.81,3.44,3.57,3.43,4.33,3.71,3.59,3.83,4.59


In [ ]:
# ── Cell 14: Save Each Model's CSV Individually + Ensemble ───────────────────
#
# Output files:
#   scores_XGBoost.csv
#   scores_LightGBM.csv
#   ... (one per model)
#   scores_Ensemble.csv
#   model_performance.csv
#   cv_results.csv

import os

output_files = []

# ── Per-model CSVs ─────────────────────────────────────────────────────────
for name in models.keys():
    score_col = f"{name}_Score"
    if score_col not in risk_scores_df.columns:
        continue

    model_df = df.loc[X_test.index, meta_cols].copy()
    model_df['Actual_Instability'] = y_test.values
    model_df['Risk_Score_0_10']    = risk_scores_df[score_col].values
    model_df['Risk_Probability']   = (risk_scores_df[score_col] / 10).round(4).values
    model_df['Optimal_Threshold']  = thresholds.get(name, 0.5)
    model_df['Predicted_Unstable'] = (
        model_df['Risk_Probability'] >= model_df['Optimal_Threshold']
    ).astype(int)

    # Safe filename
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')
    fname = f"scores_{safe_name}.csv"
    model_df.to_csv(fname, index=False)
    output_files.append(fname)
    print(f"  💾 {fname}")

# ── Ensemble CSV ───────────────────────────────────────────────────────────
ens_df = df.loc[X_test.index, meta_cols].copy()
ens_df['Actual_Instability'] = y_test.values
ens_df['Ensemble_Score']     = risk_scores_df['Ensemble_Score'].values
ens_df['Ensemble_Prob']      = (risk_scores_df['Ensemble_Score'] / 10).round(4).values
ens_df['Optimal_Threshold']  = round(best_ens_thresh, 3)
ens_df['Predicted_Unstable'] = (ens_df['Ensemble_Prob'] >= best_ens_thresh).astype(int)
# Include all individual scores for reference
for sc in score_cols:
    ens_df[sc] = risk_scores_df[sc].values

ens_df.to_csv('scores_Ensemble.csv', index=False)
output_files.append('scores_Ensemble.csv')
print(f"  💾 scores_Ensemble.csv")

# ── Summary CSVs ───────────────────────────────────────────────────────────
perf_df.to_csv('model_performance.csv', index=False)
cv_df.to_csv('cv_results.csv', index=False)
output_files.extend(['model_performance.csv', 'cv_results.csv'])
print(f"  💾 model_performance.csv")
print(f"  💾 cv_results.csv")

print(f"\n✅ {len(output_files)} files saved.")

  💾 scores_XGBoost.csv
  💾 scores_LightGBM.csv
  💾 scores_CatBoost.csv
  💾 scores_Random_Forest.csv
  💾 scores_Extra_Trees.csv
  💾 scores_HistGradientBoosting.csv
  💾 scores_Gradient_Boosting.csv
  💾 scores_AdaBoost.csv
  💾 scores_Ensemble.csv


NameError: name 'cv_df' is not defined

In [ ]:
# ── Cell 15: Download All Files (Colab) ──────────────────────────────────────
try:
    from google.colab import files
    print("⬇️  Downloading all files...\n")
    for fname in output_files:
        if os.path.exists(fname):
            files.download(fname)
            print(f"  ✓ {fname}")
    print("\n✅ All downloads triggered.")
except ImportError:
    print("ℹ️  Not in Colab — all files saved locally in the working directory.")
    print("   Files:", output_files)

⬇️  Downloading all files...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ scores_XGBoost.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ✓ scores_AdaBoost.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ scores_Ensemble.csv

✅ All downloads triggered.


In [ ]:
import joblib
import os

model_files = []

os.makedirs("saved_models", exist_ok=True)

for name, model in models.items():
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')
    filepath = f"saved_models/{safe_name}.pkl"

    joblib.dump(model, filepath)
    model_files.append(filepath)
    print(f"💾 Saved: {filepath}")

💾 Saved: saved_models/XGBoost.pkl
💾 Saved: saved_models/LightGBM.pkl
💾 Saved: saved_models/CatBoost.pkl
💾 Saved: saved_models/Random_Forest.pkl
💾 Saved: saved_models/Extra_Trees.pkl
💾 Saved: saved_models/HistGradientBoosting.pkl
💾 Saved: saved_models/Gradient_Boosting.pkl
💾 Saved: saved_models/AdaBoost.pkl


In [ ]:
ensemble_data = {
    "model_names": list(models.keys()),
    "thresholds": thresholds,
    "best_ensemble_threshold": best_ens_thresh,
    "score_columns": score_cols
}

joblib.dump(ensemble_data, "saved_models/ensemble.pkl")
print("💾 Saved: saved_models/ensemble.pkl")

💾 Saved: saved_models/ensemble.pkl


In [ ]:
full_bundle = {
    "models": models,
    "thresholds": thresholds,
    "ensemble_threshold": best_ens_thresh
}

joblib.dump(full_bundle, "saved_models/full_pipeline.pkl")
print("💾 Saved: saved_models/full_pipeline.pkl")

💾 Saved: saved_models/full_pipeline.pkl


In [ ]:
import shutil

shutil.make_archive("saved_models", 'zip', "saved_models")
from google.colab import files
files.download("saved_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>